<a href="https://colab.research.google.com/github/GilliardMorandim/mba-tcc-usp-inadimplencia/blob/eda%2Ffeature/pre_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

In [42]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark
!pip install mlxtend


In [43]:
import matplotlib.pyplot as plt
from google.colab import drive
import os
import pandas as pd

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from mlxtend.feature_selection import SequentialFeatureSelector as sfs
from sklearn.linear_model import LinearRegression

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Lendo Arquivo Pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when


spark = (SparkSession.builder
         .appName("LoadAllCSVs")
         .config("spark.driver.memory", "8g")
         .config("spark.executor.memory", "8g")
         .config("spark.sql.files.maxPartitionBytes", "256m")
         .getOrCreate())

print("Spark iniciado!")

Spark iniciado!


In [4]:
dfs_spark = {}

base_path = "/content/drive/MyDrive/MBA - Ciencia de Dados - USP/dados_tcc/"

files = [f for f in os.listdir(base_path) if f.endswith(".csv")]

for file in files:
    full_path = os.path.join(base_path, file)
    print(f"\n📥 Lendo via PySpark: {file}")

    try:
        df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .csv(full_path))

        key = file.replace(".csv", "")
        dfs_spark[key] = df

        print(f"✔ OK - Linhas (estimado pela Spark): {df.count()} | Colunas: {len(df.columns)}")

    except Exception as e:
        print(f"❌ Erro ao ler {file}: {e}")

print("\nTodos os arquivos foram processados!")

globals().update(dfs_spark)


📥 Lendo via PySpark: pre_aprovado.csv
✔ OK - Linhas (estimado pela Spark): 42171363 | Colunas: 11

📥 Lendo via PySpark: parcelas.csv
✔ OK - Linhas (estimado pela Spark): 1865444 | Colunas: 21

📥 Lendo via PySpark: contratos.csv
✔ OK - Linhas (estimado pela Spark): 382539 | Colunas: 26

📥 Lendo via PySpark: score_credito.csv
✔ OK - Linhas (estimado pela Spark): 266126 | Colunas: 3

📥 Lendo via PySpark: analise_conversao.csv
✔ OK - Linhas (estimado pela Spark): 538023 | Colunas: 12

📥 Lendo via PySpark: analise_conversao_v2.csv
✔ OK - Linhas (estimado pela Spark): 538023 | Colunas: 12

📥 Lendo via PySpark: analise_conversao_v3.csv
✔ OK - Linhas (estimado pela Spark): 533525 | Colunas: 11

Todos os arquivos foram processados!


# Pre-Processing

In [26]:
parcelas_main = parcelas

contratos_keys = contratos.select(
    "id_contrato",
    "uuid_cliente",
    "id_contrato_original",
    "id_contrato_pai",
    "cpf_hash_sha256").filter("id_contrato IS NOT NULL")


parcelas_main = parcelas.join(
    contratos_keys,
    on="id_contrato",
    how="left"
)

parcelas_main = parcelas_main.filter("data_vencimento < '2025-11-01'")

# Normalização da parcela

In [27]:
# janela por contrato
w = Window.partitionBy("id_contrato")

parcelas_main = (
    parcelas_main

    .withColumn("min_parcela", F.min("parcela").over(w))
    .withColumn("max_parcela", F.max("parcela").over(w))
    .withColumn(
        "parcela_norm_0_1",
        F.round((F.col("parcela") - F.col("min_parcela")) /
        (F.col("max_parcela") - F.col("min_parcela")),2
    ))
    .drop("min_parcela", "max_parcela")
)

#parcelas_main.orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)
#parcelas_main.show(10,truncate=False)

In [28]:
#define a janela de particionamento de underbound
w_ffill =(Window.partitionBy("id_contrato")
          .orderBy("parcela")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow))

parcelas_main = parcelas_main.withColumn("data_pagamento_aux",F.last("data_pagamento",ignorenulls=True).over(w_ffill))

parcelas_main = parcelas_main.withColumn("data_vencimento_aux",F.greatest
 (F.datediff(F.col("data_pagamento_aux"),F.col("data_vencimento")), F.lit(0)))
parcelas_main = parcelas_main.drop("data_vencimento_aux")

In [29]:
#parcelas_main.orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)

# Flag Contrato sem nenhum pagamento

In [30]:
w_contrato = Window.partitionBy("id_contrato")

parcelas_main = parcelas_main.withColumn(
    "flag_contrato_sem_pagamento",
    F.when(
        F.max(F.col("data_pagamento").isNotNull().cast("int")).over(w_contrato) == 0,
        F.lit(1)
    ).otherwise(F.lit(0))
)


In [31]:
#Validando flag-inadimplencia over todas parcelas
parcelas_main.select(
    "id_contrato",
    "parcela",
    "data_vencimento",
    "data_pagamento",
    "data_pagamento_aux",
    "flag_contrato_sem_pagamento",
).orderBy("data_vencimento").filter(F.col("id_contrato") == "{00012C6D-DD36-401A-8B26-F0C46C121114}").show(100, truncate=False)

+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+
|id_contrato                           |parcela|data_vencimento|data_pagamento|data_pagamento_aux|flag_contrato_sem_pagamento|
+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+
|{00012C6D-DD36-401A-8B26-F0C46C121114}|1      |2025-07-25     |NULL          |NULL              |1                          |
|{00012C6D-DD36-401A-8B26-F0C46C121114}|2      |2025-08-25     |NULL          |NULL              |1                          |
|{00012C6D-DD36-401A-8B26-F0C46C121114}|3      |2025-09-25     |NULL          |NULL              |1                          |
+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+



# Data Pagamento Aux

In [32]:
w_primeira = (
    Window
    .partitionBy("id_contrato")
    .orderBy("parcela")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

# Criação da coluna com a data de vencimento da primeira parcela
parcelas_main = parcelas_main.withColumn(
    "data_vencimento_primeira_parcela",
    F.first("data_vencimento", ignorenulls=True).over(w_primeira)
)


In [33]:
parcelas_main = parcelas_main.withColumn(
    "qtd_dias_de_atraso_v2",
    F.when(
        # 🔴 Cenário B — contrato nunca teve pagamento
        F.col("flag_contrato_sem_pagamento") == 1,
        F.greatest(
            F.datediff(
                F.col("data_vencimento"),
                F.col("data_vencimento_primeira_parcela")
            ),
            F.lit(0)
        )
    ).when(
        # 🟢 Cenário A — contrato teve pagamento
        (F.col("data_pagamento_aux").isNotNull()) &
        (F.col("data_vencimento") > F.col("data_pagamento_aux")),
        F.abs(
            F.datediff(
                F.col("data_vencimento"),
                F.col("data_pagamento_aux")
            )
        )
    ).otherwise(F.lit(0))
)

#Flag inadimplência

In [34]:

parcelas_main = parcelas_main.withColumn(
    "flag_inadimplencia_30_days",
    when(col("qtd_dias_de_atraso_v2")>30,1).otherwise(0)).withColumn(
     "flag_inadimplencia_90_days",when(col("qtd_dias_de_atraso_v2")>90,1).otherwise(0))

#parcelas_main.show(5, truncate=False)

In [35]:
#Validando flag-inadimplencia over todas parcelas
parcelas_main.select(
    "id_contrato",
    "parcela",
    "data_vencimento",
    "data_pagamento",
    "data_pagamento_aux",
    "flag_contrato_sem_pagamento",
    "qtd_dias_de_atraso_v2",
    "flag_inadimplencia_30_days",
    "flag_inadimplencia_90_days"
).orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)
#{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}
#{00012C6D-DD36-401A-8B26-F0C46C121114}

+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+---------------------+--------------------------+--------------------------+
|id_contrato                           |parcela|data_vencimento|data_pagamento|data_pagamento_aux|flag_contrato_sem_pagamento|qtd_dias_de_atraso_v2|flag_inadimplencia_30_days|flag_inadimplencia_90_days|
+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+---------------------+--------------------------+--------------------------+
|{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}|1      |2024-11-27     |2024-11-17    |2024-11-17        |0                          |10                   |0                         |0                         |
|{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}|2      |2024-12-27     |NULL          |2024-11-17        |0                          |40                   |1                         |0            

In [36]:
parcelas_main.orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)

+--------------------------------------+--------------+---------------+------+---------+------------------------------+------------+-------------+------------+-----------+-----------+-----------------+----------+--------------+------------------+--------------------------+----------------+--------------------------------------+-------+------------------+-------------------------------+--------------------------------------+--------------------------------------+---------------+----------------------------------------------------------------+----------------+------------------+---------------------------+--------------------------------+---------------------+--------------------------+--------------------------+
|id_contrato                           |data_pagamento|data_vencimento|valor |valor_iof|valor_financiado_principal_iof|qtd_parcelas|valor_parcela|valor_tarifa|valor_juros|valor_iof_2|valor_amortizacao|valor_pago|valor_desconto|valor_juros_atraso|valor_juros_remuneratorios|status

# Extrair Mes e Ano do Vencimento da Parcela

In [37]:
parcelas_main = parcelas_main.withColumn('year',F.year(F.to_timestamp('data_vencimento', 'yyyy-MM-dd')))
parcelas_main = parcelas_main.withColumn('month',F.month(F.to_timestamp('data_vencimento', 'yyyy-MM-dd')))

## PCT - Juros e Amortização

In [38]:
from pyspark.sql.functions import try_divide

parcelas_main = parcelas_main.withColumn(
    "pct_juros",
    F.coalesce(
        F.round(try_divide(F.col("valor_juros"), F.col("valor_parcela")), 2),
        F.lit(0)
    ))

parcelas_main = parcelas_main.withColumn(
    "pct_amortizacao",
    F.coalesce(
        F.round(try_divide(F.col("valor_amortizacao"), F.col("valor_parcela")), 2),
        F.lit(0)
    ))

In [40]:
#spark.conf.set("spark.sql.ansi.enabled", "false")
#parcelas_main.show(100, truncate=False)

# Feature Selection

================== # VARIAVEIS DE ALAVANCAGEM # ========================

flag_contrato_sem_pagamento (Avaliar se o não pagamento da primeira parcela acarreta em inadimplência)

qtd_parcelas

valor

valor_parcela

valor_financiado_principal_iof

valor_iof

valor_tarifa

=====================================================================

================== # VARIAVEIS DE SAZONALIDADE # ========================

parcela_norm_0_1

mes_vencimento * transformar a variael

ano_vencimento * transformar a variavel

=====================================================================

================== # VARIAVEIS DE DECOMPOSIÇÃO FINANCEIRA # ========================

valor_amortizacao

valor_juros

valor_juros_remuneratorios

valor_iof_2

E razões (features derivadas):

* pct_juros = valor_juros / valor_parcela

* pct_amortizacao = valor_amortizacao / valor_parcela

Justificativa: Parcelas mais “carregadas de juros” tendem a inadimplir mais.

=====================================================================


================== # VARIAVEIS DE DECOMPOSIÇÃO FINANCEIRA # =============



In [ ]:
#https://archive.is/03ptu


# Foward Selection

In [44]:
lreg = LinearRegression()
sfs1 = sfs(lreg, k_features=4, forward=True, verbose=2,scoring="neg_mean_squared_error" )

In [48]:
parcelas_main_pd = parcelas_main.toPandas()
X = parcelas_main_pd.drop("flag_inadimplencia_30_days", "flag_inadimplencia_90_days")
y = parcelas_main_pd["flag_inadimplencia_90_days"]

TypeError: DataFrame.drop() takes from 1 to 2 positional arguments but 3 were given

In [ ]:
sfs1 = sfs1.fit(X, y)

feat_names = list(sfs1.k_feature_names_)
print(feat_names)

In [ ]:
X.head(5)